# Domain 3: Claude Code Configuration & Workflows
## TS 3.3: Apply Path-Specific Rules for Conditional Convention Loading
**Certification Focus**: CCAR-F Domain 3 (20% Exam Weight) | **Scenario Alignment**: Scenario 2 (Code Generation with Claude Code)

---

### 🎯 Key Architectural Principles
1. **Dynamic Path Scoping**: Instead of packing every project rule into a single bloated `CLAUDE.md`, rules should live in `.claude/rules/<name>.md` with frontmatter `paths: [...]`.
2. **Conditional Convention Loading**: Rules are dynamically loaded into Claude's prompt context **only when** modified or targeted files match the glob patterns.
3. **Preventing Instruction Conflict**: Frontend styling conventions should not pollute backend SQL query development; path scoping ensures domain guidelines remain separated.

In [ ]:
# Step 1: Environment & Setup
import fnmatch
import re
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 3.3 Path-Specific Rules.")

### 🎯 Step 2: Rule Engine Implementation
The `RuleEngine` stores rules with glob patterns and returns only instructions applicable to currently touched files.

In [ ]:
# Step 2: Scoped Rule Engine
@dataclass
class ScopedRule:
    rule_id: str
    paths: List[str]
    instructions: str

    def matches_files(self, touched_files: List[str]) -> bool:
        for f in touched_files:
            clean_path = f.replace("\\", "/")
            for p in self.paths:
                clean_p = p.replace("\\", "/")
                if fnmatch.fnmatch(clean_path, clean_p) or fnmatch.fnmatch(clean_path, f"*/{clean_p}"):
                    return True
                if clean_p.endswith("/**"):
                    prefix = clean_p[:-3]
                    if clean_path.startswith(prefix) or f"/{prefix}" in clean_path:
                        return True
        return False


class RuleEngine:
    def __init__(self) -> None:
        self.rules: List[ScopedRule] = []

    def register_rule(self, rule_id: str, raw_markdown: str) -> None:
        paths: List[str] = []
        instructions = raw_markdown
        match = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)$", raw_markdown, re.DOTALL)
        if match:
            frontmatter, instructions = match.groups()
            for line in frontmatter.splitlines():
                line = line.strip()
                if line.startswith("paths:"):
                    continue
                if line.startswith("- "):
                    val = line[2:].replace("'", "").replace('"', "").strip()
                    if val:
                        paths.append(val)
        self.rules.append(ScopedRule(rule_id=rule_id, paths=paths, instructions=instructions.strip()))

    def get_applicable_instructions(self, touched_files: List[str]) -> List[Dict[str, Any]]:
        applicable = []
        for rule in self.rules:
            if rule.matches_files(touched_files):
                applicable.append({
                    "rule_id": rule.rule_id,
                    "scoped_patterns": rule.paths,
                    "instructions": rule.instructions,
                })
        return applicable

print(" Scoped RuleEngine ready.")

### 🧪 Step 3: Interactive Simulation
We register backend and frontend rules and inspect which instructions activate when different files are touched.

In [ ]:
# Step 3: Simulation
engine = RuleEngine()

# Register Backend Rule
engine.register_rule(
    "backend_api_rule",
    """---
paths:
  - "backend/**"
  - "*.py"
---
Use FastAPI dependency injection and explicit Pydantic response models.
"""
)

# Register Frontend Rule
engine.register_rule(
    "frontend_ui_rule",
    """---
paths:
  - "frontend/**"
  - "*.tsx"
---
Use Tailwind CSS classes and React functional components with hooks.
"""
)

# Scenario A: Editing backend file
backend_active = engine.get_applicable_instructions(["backend/routers/orders.py"])
print("Active rules for backend/routers/orders.py:")
for r in backend_active:
    print(f"  [{r['rule_id']}]: {r['instructions']}")

# Scenario B: Editing frontend file
frontend_active = engine.get_applicable_instructions(["frontend/components/Navbar.tsx"])
print("\nActive rules for frontend/components/Navbar.tsx:")
for r in frontend_active:
    print(f"  [{r['rule_id']}]: {r['instructions']}")

### 🔬 Step 4: Verification Assertions
We assert that:
1. Backend files activate backend rules and strictly exclude frontend rules.
2. Frontend files activate frontend rules and strictly exclude backend rules.

In [ ]:
# Step 4: Verification Checks
backend_ids = [r["rule_id"] for r in backend_active]
assert "backend_api_rule" in backend_ids
assert "frontend_ui_rule" not in backend_ids

frontend_ids = [r["rule_id"] for r in frontend_active]
assert "frontend_ui_rule" in frontend_ids
assert "backend_api_rule" not in frontend_ids

print(" All verification assertions passed! Path-scoped rules validated.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Keep root `CLAUDE.md` concise (<200 lines); offload domain-specific conventions to `.claude/rules/*.md`.
- **Rule 2**: Use glob matching (`paths: ["src/api/**"]`) in frontmatter to conditionally load instructions.
- **Rule 3**: Scoping prevents conflicting instructions between disparate sub-stacks (e.g. frontend vs backend).